## Scale-up study — SnapUGC watch-retention transfer at higher data volume

**Question.** Does giving the transfer head *more* data change the honest-negative result? The first pass used only **150** SnapUGC videos. Here we scale to **10,137** (**68×**) and re-measure.

**Data (legally, no block-evasion).** SnapUGC (ECCV 2024) ships as a CSV of per-video Snapchat-CDN links + an **ECR** label; ~50% of the 2024 links still resolve. Pulled with a multithreaded downloader (~35 videos/s, ~40 MB/s) → **10,137 videos, ~15 GB**.

**Features.** parallel video decode (8 threads, cv2/ffmpeg) + serial MPS encode → CLIP-512+SigLIP-768+CLAP-512 = **1792-d** per video.

> *Multithreading note:* download is network-I/O → threads gave ~24×. Feature extraction is single-GPU (MPS) → threads only parallelise video decode, so ~2–4×, not 24×.


### Results

| SnapUGC videos (head) | head Spearman (ECR) | breakout Δ (mm_best → +retention) | ER Δ |
|---|---|---|---|
| 150 | 0.317 | 0.6539 → 0.6533 (**-0.0006**) | 0.4969 → 0.4962 (**-0.0007**) |
| 10,137 | 0.451 | 0.6539 → 0.6409 (**-0.013**) | 0.4969 → 0.4936 (**-0.0033**) |

**Result at 10,137 videos:** breakout Δ=-0.013, ER Δ=-0.0033. The extra data **moves the needle** — see the table.


### Reproduce (loads the ledger `reports/snapugc_scaleup.json`)


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
d = json.load(open('../reports/snapugc_scaleup.json'))
runs = sorted(d['ablation']['runs'], key=lambda x: x['n_snapugc_head'])
df = pd.DataFrame([{'n_snapugc': r['n_snapugc_head'],
                    'head_spearman_ecr': r['head_spearman_ecr'],
                    'breakout_delta': r['breakout']['delta'],
                    'er_delta': r['er']['delta']} for r in runs])
display(df)
ax = df.set_index('n_snapugc')[['breakout_delta','er_delta']].plot.bar(rot=0)
ax.axhline(0, color='k', lw=0.8); ax.set_ylabel('LOCO AUC lift from retention_head')
ax.set_title('SnapUGC retention transfer: lift vs head-training size'); plt.tight_layout(); plt.show()


**Takeaway.** Result at 10,137 videos: breakout Δ=-0.013, ER Δ=-0.0033. The extra data moves the needle — see the table.
